# PEFT(Parameter-Efficient Fine-Tuning)

https://huggingface.co/docs/peft/en/index

PEFT(Parameter-Efficient Fine-Tuning)는 대형 사전학습 모델의 대부분 파라미터를 고정한 상태에서, 일부 파라미터만 추가하거나 수정하여 특정 작업에 맞게 모델을 적응시키는 파인튜닝 방식이다.

전체 모델을 다시 학습하는 Full Fine-tuning에 비해 학습 비용과 저장 공간을 크게 줄일 수 있으며, 여러 작업별 어댑터를 따로 저장하고 교체할 수 있다는 장점이 있다.

## PEFT의 핵심 장점

- 학습해야 하는 파라미터 수 감소
- GPU 메모리 사용량 절감
- 학습 시간 및 비용 절감
- 원본 모델의 일반 지식 유지
- 작업별 어댑터 저장 및 교체 가능
- 일부 방식은 추론 시 병합하여 추가 지연을 줄일 수 있음

> PEFT는 단순히 “가볍게 학습하는 기술”이 아니라, 대형 모델을 현실적인 자원 안에서 특정 작업에 맞게 조정하기 위한 핵심 방법론이다.

---

# 1. PEFT 기법의 큰 분류

PEFT 기법은 모델에 개입하는 방식에 따라 크게 세 가지로 나눌 수 있다.

| 분류 | 핵심 아이디어 | 대표 기법 |
|---|---|---|
| Additive Methods | 기존 모델은 고정하고 새로운 구성요소를 추가 | Prompt-tuning, Prefix-tuning, Adapters |
| Selective Methods | 기존 파라미터 중 일부만 선택적으로 학습 | BitFit, Top-layer Fine-tuning |
| Reparameterization Methods | 가중치 변화를 저차원 구조로 표현 | LoRA, QLoRA, AdaLoRA, DoRA |

---

# 2. Additive Methods: 추가 파라미터 방식

Additive Methods는 기존 모델의 가중치는 고정하고, 학습 가능한 작은 구성요소를 새로 추가하는 방식이다.

## 2-1. Prompt-tuning

Prompt-tuning은 입력 임베딩 앞에 학습 가능한 가상 토큰(Soft Prompt)을 추가하는 방식이다.

- 모델 본체는 수정하지 않는다.
- 입력층 근처에만 학습 가능한 벡터를 추가한다.
- 구현이 단순하고 저장 공간이 작다.
- 모델 크기가 클수록 효과가 좋아지는 경향이 있다.
- 빠른 프로토타이핑이나 태스크별 프롬프트 학습에 적합하다.

단점은 모델 내부 표현을 직접 조정하지 않기 때문에, 복잡한 작업에서는 LoRA 계열보다 성능이 낮을 수 있다는 점이다.

## 2-2. P-tuning

P-tuning은 단순히 가상 토큰을 나열하는 대신, Prompt Encoder를 사용해 더 유연한 프롬프트 표현을 생성하는 방식이다.

- 가상 토큰 사이의 관계를 더 잘 반영할 수 있다.
- 문장 앞부분뿐 아니라 중간 위치에도 가상 토큰을 넣는 방식으로 확장될 수 있다.
- Prompt-tuning보다 표현력이 높지만 구조가 조금 더 복잡하다.

## 2-3. Prefix-tuning

Prefix-tuning은 각 Transformer 레이어의 Key, Value 앞에 학습 가능한 Prefix 벡터를 붙이는 방식이다.

- 입력층뿐 아니라 모델 내부 연산에도 영향을 준다.
- Prompt-tuning보다 안정적인 성능을 보이는 경우가 많다.
- 단, Prefix 길이만큼 사용 가능한 컨텍스트 길이가 줄어들 수 있다.
- 추론 시 시퀀스 길이 점유가 발생할 수 있다.

## 2-4. Adapters

Adapters는 Transformer 레이어 사이에 작은 병목 구조의 신경망을 삽입하고, 이 어댑터만 학습하는 방식이다.

- 원본 모델 파라미터는 고정한다.
- 각 레이어 뒤에 작은 Feed-forward 모듈을 추가한다.
- 다양한 태스크별 어댑터를 따로 저장할 수 있다.
- 안정적인 성능을 기대할 수 있다.

단점은 모델 구조에 추가 모듈이 들어가기 때문에, 추론 시 약간의 레이턴시 오버헤드가 생길 수 있다는 점이다.

---

# 3. Selective Methods: 선택적 파인튜닝 방식

Selective Methods는 기존 모델의 파라미터 중 일부만 선택적으로 학습하는 방식이다.

## 3-1. BitFit

BitFit은 모델의 bias 파라미터만 학습하는 방식이다.

- 학습 파라미터 수가 매우 적다.
- 구현이 단순하다.
- 빠른 실험에 적합하다.
- 저장 공간이 매우 작다.

다만 학습 가능한 범위가 매우 제한적이므로, 복잡한 태스크에서는 성능 한계가 뚜렷할 수 있다.

## 3-2. Top-layer Fine-tuning

Top-layer Fine-tuning은 모델의 상위 레이어 일부만 학습하는 방식이다.

- 하위 레이어는 일반적인 언어 지식 표현을 유지한다.
- 상위 레이어에서 태스크 특화 표현을 학습한다.
- 간단한 분류나 도메인 적응에 사용할 수 있다.

하지만 대형 LLM에서는 LoRA 계열이 더 널리 사용되는 편이다.

---

# 4. Reparameterization Methods: 재매개변수화 방식

Reparameterization Methods는 기존 가중치를 직접 모두 수정하지 않고, 가중치 변화량을 더 작은 구조로 표현하는 방식이다.

현재 LLM 파인튜닝에서 가장 널리 사용되는 계열은 LoRA 계열이다.

## 4-1. LoRA(Low-Rank Adaptation)

LoRA는 기존 가중치 행렬 `W`를 직접 수정하지 않고, 가중치 변화량 `ΔW`를 두 개의 작은 저랭크 행렬 `A`, `B`로 표현하는 방식이다.

$$
W' = W + \Delta W
$$

$$
\Delta W = A \times B
$$

여기서 `A`, `B`는 원래 가중치보다 훨씬 작은 행렬이므로 학습해야 하는 파라미터 수가 크게 줄어든다.

### LoRA의 장점

- 학습 파라미터 수가 적다.
- 원본 모델을 고정한 채 작업별 어댑터만 학습할 수 있다.
- 추론 시 LoRA 가중치를 원본 모델에 병합할 수 있다.
- 병합 후에는 추가 추론 레이턴시가 거의 없다.
- 구현과 운영이 비교적 단순하다.

### LoRA가 기본 선택지로 많이 쓰이는 이유

- 성능과 효율의 균형이 좋다.
- Hugging Face PEFT 생태계에서 지원이 안정적이다.
- QLoRA, AdaLoRA, DoRA 등 여러 확장 기법의 기반이 된다.

## 4-2. QLoRA(Quantized LoRA)

QLoRA는 모델 본체를 4비트로 양자화한 뒤, 그 위에 LoRA 어댑터를 학습하는 방식이다.

- 원본 모델은 4-bit로 로드한다.
- 학습은 LoRA 어댑터 중심으로 진행한다.
- NF4 양자화와 Paged Optimizer를 함께 사용하는 경우가 많다.
- VRAM이 제한된 환경에서 대형 모델을 학습할 수 있게 해준다.

### QLoRA의 장점

- 매우 적은 VRAM으로 대형 모델 학습 가능
- 소비자용 GPU에서도 실습 가능
- LoRA의 장점과 양자화의 장점을 함께 활용

### QLoRA를 선택하기 좋은 상황

- GPU VRAM이 부족한 경우
- 7B, 8B, 13B 이상의 모델을 제한된 GPU에서 학습해야 하는 경우
- 수업이나 실험 환경에서 비용을 줄이고 싶은 경우

## 4-3. AdaLoRA(Adaptive LoRA)

AdaLoRA는 LoRA의 rank를 모든 레이어에 동일하게 두지 않고, 중요도에 따라 적응적으로 배분하는 방식이다.

- 중요한 가중치 행렬에는 더 높은 rank를 할당한다.
- 덜 중요한 가중치 행렬에는 낮은 rank를 할당한다.
- 같은 파라미터 예산 안에서 효율을 높이는 것을 목표로 한다.

다만 기본 LoRA보다 설정과 이해가 더 복잡하므로, 처음 실습에서는 LoRA나 QLoRA를 먼저 사용하는 것이 좋다.

## 4-4. DoRA(Weight-Decomposed LoRA)

DoRA는 가중치 변화를 크기(Magnitude)와 방향(Direction)으로 나누어 다루는 LoRA 확장 방식이다.

- 가중치의 방향 성분에는 LoRA를 적용한다.
- 크기 성분은 별도로 조정한다.
- LoRA보다 더 높은 성능을 보이는 경우가 있다.
- Full Fine-tuning에 더 가까운 학습 패턴을 목표로 한다.

다만 일반 수업이나 초급 실습에서는 LoRA와 QLoRA를 먼저 다룬 뒤, 고성능 확장 기법으로 소개하는 것이 적절하다.

## 4-5. IA³

IA³는 활성화 값에 학습 가능한 스케일 벡터를 곱해 모델을 조정하는 방식이다.

- 학습 파라미터 수가 매우 적다.
- 추론 오버헤드가 작다.
- 자원이 극도로 제한된 환경에 적합하다.
- 표현력은 LoRA 계열보다 제한적일 수 있다.

---

# 5. PEFT 기법 비교

아래 표는 각 기법의 일반적인 특징을 비교한 것이다.  
정확한 성능은 모델 크기, 데이터셋, 태스크, 학습 설정에 따라 달라질 수 있으므로 절대적인 수치가 아니라 선택 기준으로 이해하는 것이 좋다.

| 기법 | 학습 파라미터 규모 | 추론 오버헤드 | 장점 | 권장 상황 |
|---|---:|---|---|---|
| Prompt-tuning | 매우 작음 | 거의 없음 | 구현 단순, 빠른 실험 | 빠른 프로토타입 |
| P-tuning | 작음 | 거의 없음 | 프롬프트 표현력 향상 | 유연한 프롬프트 학습 |
| Prefix-tuning | 작음 | 시퀀스 길이 점유 | 비교적 안정적인 성능 | 생성 태스크 실험 |
| BitFit | 매우 작음 | 없음 | 가장 단순한 선택적 학습 | 극소 자원 실험 |
| IA³ | 매우 작음 | 거의 없음 | 파라미터 효율 매우 높음 | 자원 제한 환경 |
| Adapters | 중간 | 약간 있음 | 안정적인 태스크 적응 | 여러 태스크 어댑터 관리 |
| LoRA | 작음 | 병합 시 없음 | 성능과 효율의 균형 | 기본 선택지 |
| QLoRA | 작음 | 병합/로드 방식에 따라 다름 | VRAM 절감 효과 큼 | 제한된 GPU 환경 |
| AdaLoRA | 작음 | 병합 시 없음 | rank를 적응적으로 배분 | 효율 최적화 |
| DoRA | 작음~중간 | 병합 시 없음 | LoRA보다 높은 성능 가능 | 고성능 튜닝 |
| Full Fine-tuning | 전체 | 없음 | 가장 직접적인 학습 | 자원이 충분한 경우 |

---

# 6. 가상 토큰 방식 vs 가중치 수정 방식

## 가상 토큰 방식

Prompt-tuning, P-tuning, Prefix-tuning은 모델의 가중치를 직접 바꾸기보다, 입력이나 내부 Key/Value 앞에 학습 가능한 가상 토큰을 추가하는 방식이다.

### 장점

- 구현이 단순하다.
- 원본 모델 구조를 크게 바꾸지 않는다.
- 저장해야 할 파라미터가 매우 적다.

### 단점

- Prefix-tuning의 경우 컨텍스트 길이를 일부 차지할 수 있다.
- 복잡한 도메인 적응에서는 LoRA 계열보다 성능이 낮을 수 있다.
- 모델 크기와 태스크에 따라 성능 편차가 크다.

## 가중치 수정 방식

LoRA, QLoRA, AdaLoRA, DoRA는 모델의 가중치 변화량을 효율적으로 표현하는 방식이다.

### 장점

- 모델 내부 표현을 직접 조정할 수 있다.
- 성능이 안정적인 편이다.
- LoRA 계열은 추론 시 병합이 가능하다.
- 병합 후에는 추가 레이턴시 없이 사용할 수 있다.

### 단점

- Prompt-tuning보다 구현 개념이 조금 더 복잡하다.
- 어떤 레이어에 적용할지, rank를 얼마로 둘지 등의 설정이 필요하다.

> 현재 LLM 파인튜닝 실무에서는 성능, 구현 편의성, 생태계 지원을 고려할 때 LoRA 계열이 가장 기본적인 선택지로 사용된다.

---

# 7. 실용적 선택 가이드

## 상황별 추천

| 상황 | 추천 기법 | 이유 |
|---|---|---|
| 기본 파인튜닝 실습 | LoRA | 구현이 쉽고 성능이 안정적이다. |
| GPU VRAM이 부족함 | QLoRA | 4비트 양자화로 메모리 사용량을 크게 줄일 수 있다. |
| 최고 성능을 더 추구함 | DoRA, AdaLoRA | LoRA보다 더 정교한 조정이 가능하다. |
| 빠른 실험이 필요함 | BitFit, Prompt-tuning | 구조가 단순하고 반복 실험이 빠르다. |
| 저장 공간이 매우 제한됨 | IA³, BitFit | 저장해야 할 파라미터가 매우 적다. |
| 여러 태스크를 분리 관리 | Adapters, LoRA | 태스크별 어댑터를 저장하고 교체하기 쉽다. |
| 실시간 추론이 중요함 | LoRA 병합, BitFit | 추론 오버헤드를 최소화할 수 있다. |

## 데이터 특성별 추천

| 데이터 상황 | 추천 기법 | 이유 |
|---|---|---|
| 데이터가 매우 적음 | Prompt-tuning, BitFit | 과적합 위험을 줄이고 빠르게 실험할 수 있다. |
| 일반 도메인, 적은 레이블 | LoRA | 안정적인 기본 선택지이다. |
| 도메인 차이가 큼 | LoRA, DoRA | 모델 내부 표현을 더 강하게 조정할 수 있다. |
| Few-shot 성격이 강함 | Prompt-tuning, Prefix-tuning | 소량 데이터 기반 프롬프트 적응에 적합하다. |
| 대형 모델을 제한된 GPU에서 학습 | QLoRA | VRAM 절감 효과가 크다. |

---

# 8. 기법 선택 의사결정 프레임워크

```text
1. 자원 제약 확인
   ├─ VRAM이 부족함 → QLoRA
   ├─ VRAM이 충분함 → LoRA 또는 DoRA
   └─ 저장 공간이 매우 제한됨 → IA³, BitFit

2. 성능 요구사항 확인
   ├─ 최고 성능이 필요함 → DoRA, AdaLoRA
   ├─ 안정적인 성능이 필요함 → LoRA
   └─ 빠른 실험이 필요함 → BitFit, Prompt-tuning

3. 추론 레이턴시 요구사항 확인
   ├─ 실시간 추론이 중요함 → LoRA 병합, BitFit
   ├─ 약간의 오버헤드 허용 → Adapters
   └─ 제약이 크지 않음 → 대부분의 PEFT 기법 가능

4. 데이터 특성 확인
   ├─ Few-shot 중심 → Prompt-tuning, Prefix-tuning
   ├─ 도메인 차이가 큼 → LoRA, DoRA
   └─ 일반적인 SFT 실습 → LoRA 또는 QLoRA
```
---

**현재 LLM 파인튜닝에서 가장 실용적인 기본 선택지는 LoRA 계열이다.**

기본 선택: LoRA
VRAM 제약: QLoRA
고성능 확장: DoRA 또는 AdaLoRA
빠른 실험: BitFit 또는 Prompt-tuning
극소 파라미터: IA³

핵심 원칙: 파라미터 효율성과 성능의 균형을 맞추되, 추론 레이턴시와 운영 환경까지 함께 고려하여 기법을 선택해야 한다.